# 12장. LLM이 만든 분석 코드를 검증하는 방법

이 노트북은 `book/chapters/ch12_report_generation.md`의 검증 흐름을 실행합니다.

핵심 원칙은 다음과 같습니다.

- 생성 코드는 검토되지 않은 초안으로 취급합니다.
- 실제 데이터 구조와 키 관계를 먼저 확인합니다.
- 완료 주문 기준과 총합 대조처럼 분석 규칙을 명시합니다.
- 외부 통신·파일 변경·명령 실행은 승인 전에 실행하지 않습니다.
- 정적 점검 결과가 비어 있어도 안전하다고 단정하지 않습니다.


## 0. 실행 전 확인

5장에서 생성한 다음 파일을 사용합니다.

- `data/processed/customers_clean.csv`
- `data/processed/products_clean.csv`
- `data/processed/orders_clean.csv`
- `data/processed/order_items_clean.csv`

파일이 없다면 프로젝트 루트에서 `python scripts/preprocess_data.py`를 먼저 실행합니다.


## 1. 프로젝트 경로와 패키지 설정


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("프로젝트 루트:", PROJECT_ROOT.resolve())
print("전처리 데이터:", PROCESSED_DIR.resolve())
print("보고서 폴더:", REPORT_DIR.resolve())


In [ ]:
from src.llm_code_validation import (
    DEFAULT_STATIC_SCAN_EXAMPLE,
    assert_validation_ready,
    build_code_review_checklist,
    build_dataset_inventory,
    build_error_fix_prompt_template,
    build_leakage_review_table,
    load_validation_data,
    run_llm_code_validation,
    safe_category_sales,
    safe_monthly_sales,
    scan_generated_code,
    validate_feature_list,
    validate_primary_keys,
    validate_relationship_keys,
    validate_required_columns,
)


## 2. 전처리 데이터 불러오기


In [ ]:
datasets = load_validation_data(
    processed_dir=PROCESSED_DIR,
)

customers = datasets["customers"]
products = datasets["products"]
orders = datasets["orders"]
order_items = datasets["order_items"]

for name, dataframe in datasets.items():
    print(name, dataframe.shape)


## 3. 실행 전 데이터 구조 점검

필수 컬럼이 없거나 고유 키·관계에 문제가 있으면 집계 코드를 실행하지 않습니다.


In [ ]:
inventory = build_dataset_inventory(datasets)
required_column_check = validate_required_columns(datasets)
primary_key_check = validate_primary_keys(datasets)
relationship_check = validate_relationship_keys(datasets)

display(inventory)
display(required_column_check)
display(primary_key_check)
display(relationship_check)


In [ ]:
assert_validation_ready(
    required_column_check=required_column_check,
    primary_key_check=primary_key_check,
    relationship_check=relationship_check,
)

print("필수 구조와 키 관계 점검을 통과했습니다.")


## 4. 완료 주문 기준 카테고리 집계 검증

`line_total` 전체 합계를 곧바로 회계상 순매출이라고 부르지 않습니다. 이 실습에서는 `order_status == "completed"`인 주문 상세만 포함합니다.


In [ ]:
category_sales, category_validation = safe_category_sales(
    order_items=order_items,
    products=products,
    orders=orders,
)

display(category_sales)
display(category_validation)


## 5. 완료 주문 기준 월별 집계 검증


In [ ]:
monthly_sales, monthly_validation = safe_monthly_sales(
    order_items=order_items,
    orders=orders,
)

display(monthly_sales)
display(monthly_validation)


## 6. 생성 코드 정적 점검

아래 예시는 외부 네트워크 요청과 파일 쓰기를 포함합니다. 코드를 실행하지 않고 구문 구조만 검사합니다.


In [ ]:
print(DEFAULT_STATIC_SCAN_EXAMPLE)
static_scan = scan_generated_code(
    DEFAULT_STATIC_SCAN_EXAMPLE
)
display(static_scan)


정적 점검은 위험 후보를 찾는 보조 절차입니다. 결과가 비어 있어도 다음을 사람이 다시 확인해야 합니다.

- 외부 통신
- 파일 생성·수정·삭제
- 운영체제 명령 실행
- 동적 코드 실행
- 코드에 직접 작성된 API 키·토큰·비밀번호
- 출처를 확인하지 않은 패키지 설치


## 7. 머신러닝 입력값의 데이터 누수 검토


In [ ]:
leakage_review = build_leakage_review_table()
display(leakage_review)


In [ ]:
safe_features = [
    "payment_method",
    "order_month",
    "order_dayofweek",
    "gender",
    "age",
    "city",
]

validate_feature_list(safe_features)
print("누수 금지 입력값이 포함되지 않았습니다.")


In [ ]:
dangerous_features = [
    "payment_method",
    "order_status",
    "line_total",
]

forbidden_in_example = {
    "order_status",
    "line_total",
}.intersection(dangerous_features)

print("위험 입력값 예시:", sorted(forbidden_in_example))


필수 입력값이 없을 때 목록에서 조용히 제외하면 안 됩니다. 누락된 컬럼이나 누수 위험을 확인하고 문제 정의 또는 전처리 과정을 수정합니다.


## 8. 코드 리뷰 체크리스트와 오류 수정 프롬프트


In [ ]:
code_review_checklist = build_code_review_checklist()
display(code_review_checklist)


In [ ]:
error_fix_prompt = build_error_fix_prompt_template()
print(error_fix_prompt)


## 9. 전체 검증 파이프라인 실행

공통 함수는 검증 결과와 재사용 가능한 보고서 파일을 `reports/`에 저장합니다.


In [ ]:
validation_result = run_llm_code_validation(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
)

validation_result["outputs"].keys()


In [ ]:
for name, path in validation_result["output_paths"].items():
    print(f"- {name}: {path}")


## 10. 생성 결과 확인

주요 결과에는 데이터셋 인벤토리, 필수 컬럼·고유 키·관계 점검, 완료 주문 기준 집계, 데이터 누수 검토, 정적 점검, 체크리스트와 Markdown 요약이 포함됩니다.


In [ ]:
summary_path = validation_result[
    "output_paths"
]["validation_summary"]

print(summary_path.read_text(encoding="utf-8")[:2000])


## 11. 실습 과제

1. `scan_generated_code()`에 파일 삭제 또는 `subprocess.run()`을 포함한 문자열을 넣고 탐지 결과를 확인합니다.
2. 복사한 DataFrame에서 부모 키를 중복시킨 뒤 `validate_relationship_keys()` 결과를 확인합니다.
3. 완료 주문이 아닌 상태를 포함했을 때 집계 금액이 얼마나 달라지는지 비교하되, 두 값을 같은 의미의 매출로 부르지 않습니다.
4. LLM이 제안한 코드에서 필수 컬럼을 조용히 제외하는 부분을 찾아 실패 즉시 중단 방식으로 수정합니다.
5. 생성 코드 검토 결과에 프롬프트, 수정 이유, 실행 환경과 승인자를 기록합니다.


## 12. 정리

이번 장에서는 LLM 생성 코드를 데이터 구조, 집계 논리, 머신러닝 누수, 실행 안전, 재현성 관점에서 검증했습니다.

다음 장에서는 외부 데이터 수집으로 분석 범위를 확장합니다. 외부 API와 크롤링을 사용할 때도 이용약관, 개인정보, 요청량 제한, 데이터 출처와 품질을 먼저 확인합니다.
